In [ ]:
#Libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d, PchipInterpolator

#------------------------------------------------

#Constants 
Msun_in_km = 1.474  #In units where G=c=1 (Geometrized)
hbar_c = 197.3269804  #MeV·fm
MeVfm3_to_Jm3 = 1.6022e32 
Jm3_to_1km2 = 8.234568e-39 
MeVfm3_to_1km2 = MeVfm3_to_Jm3 * Jm3_to_1km2

#------------------------------------------------

#Equations of State APR-1 and the Crust Equations  
def MDI_1(P):
    return 4.1844*(P**0.81449) + 95.00135*(P**0.31736)

def APR_1(P):
    return 0.000719964*(P**1.85898) + 108.975*(P**0.340074)

def Crust_1(P):
    return 0.00873 + 103.17338*(1-np.exp(-P/0.38527)) + 7.34979*(1-np.exp(-P/0.01211))

def Crust_2(P):
    return  0.00015 + 0.00203*(1-np.exp(-P*344827.5)) + 0.10851*(1-np.exp(-P*7692.3076))
    
def Crust_3(P):                                                             
    return 0.0000051*(1-np.exp(-P*0.2373e10)) + 0.00014*(1-np.exp(-P*0.4020e8))

#------------------------------------------------

#Energy Desnity of Pure Hadronic Matter 
def den_en_hadronic(P):
    if P < 4.1725e-08:
        return Crust_3(P)
    elif P < 9.34375e-05:
        return Crust_2(P)
    elif P < 0.184:
        return Crust_1(P)
    else:
        return APR_1(P)

#Builder function: constructs a new EOS based on the parameters a4, D, Beff_14, and Ptr
def build_den_en_hybrid(a4, D, Beff_14, Ptr, dE, xm_min=900, xm_max=2500, N=8000, check_seidov=True):

    #Build quark EoS and obtain valid pressure range
    den_en_q_of_P, (Pq_min, Pq_max) = build_den_en_q_of_P(a4, D, Beff_14, xm_min=xm_min, xm_max=xm_max, N=N)

    #Causality check (sound speed squared c_s^2 = dP/dε) 
    P_test = np.geomspace(max(Pq_min,1e-5), min(Pq_max, 3e3), 500)
    den_en = den_en_q_of_P(P_test)
    cs2 = np.gradient(P_test, den_en)   #dP/dε
    print("cs2 min/max:", np.nanmin(cs2), np.nanmax(cs2))

    #Hadronic energy density at transition pressure 
    Etr = float(APR_1(Ptr))

    #Seidov stability criterion
    if check_seidov:
        dEcrit = 0.5 * Etr + 1.5 * Ptr
        if dE > dEcrit:
            print(f"Seidov satisfied: dE={dE:.3f} > dEcrit={dEcrit:.3f} (Ptr={Ptr:.3f})")
        else:
            print(f"Seidov unsatisfied: dE={dE:.3f} <= dEcrit={dEcrit:.3f} (Ptr={Ptr:.3f})")

    #Quark energy density at transition pressure
    Eq_tr = den_en_q_of_P(Ptr)
    #Check whether quark phase exists at Ptr
    if np.isnan(Eq_tr):
        raise ValueError(f"Ptr={Ptr} out of quark EOS range [{Pq_min},{Pq_max}] for (a4,D,Beff_14)=({a4},{D},{Beff_14}).") 
    Eq_tr = float(Eq_tr)

    #Shift quark energy density to enforce desired energy density jump
    shift = (Etr + dE) - Eq_tr

    #Hybrid EoS used in TOV integration
    def den_en_hybrid_of_P(P):
        if P < 4.1725e-08:
            return Crust_3(P)
        elif P < 9.34375e-05:
            return Crust_2(P)
        elif P < 0.184:
            return Crust_1(P)
        elif P < Ptr:
            return APR_1(P)
        else:
            den_en_q = den_en_q_of_P(P)
            if np.isnan(den_en_q):
                raise ValueError(f"P={P} out of quark EOS range [{Pq_min},{Pq_max}] for (a4,D,Beff_14)=({a4},{D},{Beff_14}). ")
            return float(den_en_q) + shift

    return den_en_hybrid_of_P